# VIO Noise Characterization — universityA

Issue #2: Analyze drift statistics from real universityA trajectories to inform the noise library (issue #3).

**Data format:** `ts_seconds x y gt_x gt_y` (pixels, dpi=2.5 px/m)  
**Noise:** `noise_x = x - gt_x`, `noise_y = y - gt_y`  
**Split:** train only — val/test excluded to prevent data leakage into noise library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from statsmodels.tsa.stattools import acf

DATA_DIR = Path('~/Developer/niloc/data/universityA').expanduser()
FLOORPLAN_PATH = DATA_DIR / 'floorplan.png'
DPI = 2.5  # px/m for universityA

TRAIN_SPLIT = DATA_DIR / 'train.txt'
VAL_SPLIT   = DATA_DIR / 'val.txt'
TEST_SPLIT  = DATA_DIR / 'test.txt'

## 1. Load train-split trajectories

In [ ]:
# Read split files — these contain bare filenames without extension
def read_split(path):
    return set(Path(path).read_text().splitlines())

train_names = read_split(TRAIN_SPLIT)
val_names   = read_split(VAL_SPLIT)
test_names  = read_split(TEST_SPLIT)

excluded = val_names | test_names
print(f'Train: {len(train_names)}  Val: {len(val_names)}  Test: {len(test_names)}')

def load_trajectory(path):
    """Load a single .txt trajectory file, skipping the comment header."""
    return np.loadtxt(path, comments='#')

trajectories = {}
for txt in sorted(DATA_DIR.glob('*.txt')):
    name = txt.stem
    if name in ('train', 'val', 'test'):
        continue
    if name in excluded:
        continue
    if name not in train_names:
        continue
    trajectories[name] = load_trajectory(txt)

print(f'Loaded {len(trajectories)} train trajectories')
sample = next(iter(trajectories.values()))
print(f'Sample shape: {sample.shape}  columns: ts, x, y, gt_x, gt_y')

## 2. Compute noise signals

In [ ]:
# Column indices
TS, X, Y, GTX, GTY = 0, 1, 2, 3, 4

noise_data = {}  # name -> dict with arrays
for name, traj in trajectories.items():
    nx = traj[:, X] - traj[:, GTX]
    ny = traj[:, Y] - traj[:, GTY]
    mag = np.sqrt(nx**2 + ny**2)
    noise_data[name] = {
        'traj': traj,
        'noise_x': nx,
        'noise_y': ny,
        'magnitude': mag,
        'n_frames': len(traj),
        'final_drift': mag[-1],
        'mean_drift': mag.mean(),
        'max_drift': mag.max(),
        'p95_drift': np.percentile(mag, 95),
    }

print('Noise computed for all trajectories.')

## 3. Summary statistics

In [ ]:
stats = pd.DataFrame([
    {
        'name': name,
        'n_frames': d['n_frames'],
        'mean_drift_px': d['mean_drift'],
        'median_drift_px': np.median(d['magnitude']),
        'p95_drift_px': d['p95_drift'],
        'max_drift_px': d['max_drift'],
        'final_drift_px': d['final_drift'],
        'mean_drift_m': d['mean_drift'] / DPI,
    }
    for name, d in noise_data.items()
])

print('=== Per-trajectory drift statistics ===')
print(stats.describe().round(2).to_string())
print()
print('=== Global summary ===')
all_mags = np.concatenate([d['magnitude'] for d in noise_data.values()])
print(f'Mean |drift|:   {all_mags.mean():.1f} px  ({all_mags.mean()/DPI:.1f} m)')
print(f'Median |drift|: {np.median(all_mags):.1f} px  ({np.median(all_mags)/DPI:.1f} m)')
print(f'p95 |drift|:    {np.percentile(all_mags, 95):.1f} px  ({np.percentile(all_mags, 95)/DPI:.1f} m)')
print(f'Max |drift|:    {all_mags.max():.1f} px  ({all_mags.max()/DPI:.1f} m)')

## 4. Noise magnitude vs frame index (20 sample trajectories)

In [ ]:
rng = np.random.default_rng(42)
sample_names = rng.choice(list(noise_data.keys()), size=min(20, len(noise_data)), replace=False)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# Top: noise magnitude per trajectory
ax = axes[0]
for name in sample_names:
    mag = noise_data[name]['magnitude']
    ax.plot(mag, alpha=0.6, linewidth=0.8)
ax.axhline(all_mags.mean(), color='red', linestyle='--', linewidth=1.2, label=f'Global mean ({all_mags.mean():.0f} px)')
ax.set_xlabel('Frame index')
ax.set_ylabel('|noise| (px)')
ax.set_title('Noise magnitude over time — 20 sample train trajectories')
ax.legend()

# Bottom: final drift distribution across all train trajectories
ax2 = axes[1]
final_drifts = [d['final_drift'] for d in noise_data.values()]
ax2.hist(final_drifts, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
ax2.axvline(np.mean(final_drifts), color='red', linestyle='--', label=f'Mean ({np.mean(final_drifts):.0f} px)')
ax2.set_xlabel('Final drift magnitude (px)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of final drift across all train trajectories')
ax2.legend()

plt.tight_layout()
plt.savefig('noise_magnitude_over_time.png', dpi=150)
plt.show()

## 5. Autocorrelation of noise_x and noise_y

VIO noise is temporally correlated (drift accumulates), so we expect high autocorrelation at short lags and slow decay — unlike i.i.d. noise which would drop immediately to zero.

In [ ]:
N_LAGS = 150

# Compute ACF on the noise *increments* (first difference) for stationarity,
# and on the raw noise to show the drift accumulation structure.
acf_x_raw, acf_y_raw = [], []
acf_x_diff, acf_y_diff = [], []

for name in sample_names:
    nx = noise_data[name]['noise_x']
    ny = noise_data[name]['noise_y']
    if len(nx) > N_LAGS + 2:
        acf_x_raw.append(acf(nx, nlags=N_LAGS, fft=True))
        acf_y_raw.append(acf(ny, nlags=N_LAGS, fft=True))
        acf_x_diff.append(acf(np.diff(nx), nlags=N_LAGS, fft=True))
        acf_y_diff.append(acf(np.diff(ny), nlags=N_LAGS, fft=True))

acf_x_raw  = np.array(acf_x_raw).mean(axis=0)
acf_y_raw  = np.array(acf_y_raw).mean(axis=0)
acf_x_diff = np.array(acf_x_diff).mean(axis=0)
acf_y_diff = np.array(acf_y_diff).mean(axis=0)

lags = np.arange(N_LAGS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(lags, acf_x_raw, label='noise_x', color='steelblue')
axes[0].plot(lags, acf_y_raw, label='noise_y', color='tomato')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('ACF of raw noise (shows drift accumulation)')
axes[0].set_xlabel('Lag (frames)')
axes[0].set_ylabel('Autocorrelation')
axes[0].legend()

axes[1].plot(lags, acf_x_diff, label='diff(noise_x)', color='steelblue')
axes[1].plot(lags, acf_y_diff, label='diff(noise_y)', color='tomato')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('ACF of noise increments (stationary signal)')
axes[1].set_xlabel('Lag (frames)')
axes[1].set_ylabel('Autocorrelation')
axes[1].legend()

plt.tight_layout()
plt.savefig('noise_autocorrelation.png', dpi=150)
plt.show()

# Report autocorrelation length (lag where ACF drops below 0.5 in raw signal)
below_half_x = np.where(acf_x_raw < 0.5)[0]
below_half_y = np.where(acf_y_raw < 0.5)[0]
corr_len_x = below_half_x[0] if len(below_half_x) else N_LAGS
corr_len_y = below_half_y[0] if len(below_half_y) else N_LAGS
print(f'Autocorrelation length (ACF < 0.5): noise_x={corr_len_x} frames, noise_y={corr_len_y} frames')

## 6. Drift growth rate

In [ ]:
# Estimate drift growth rate: fit a linear trend to mean |noise| vs frame index
# across all trajectories (aligned to the same start frame)

max_frames = max(d['n_frames'] for d in noise_data.values())
padded = []
for d in noise_data.values():
    mag = d['magnitude']
    pad = np.full(max_frames, np.nan)
    pad[:len(mag)] = mag
    padded.append(pad)

padded = np.array(padded)  # (n_traj, max_frames)
mean_mag = np.nanmean(padded, axis=0)
frame_idx = np.arange(max_frames)
valid = ~np.isnan(mean_mag) & (np.sum(~np.isnan(padded), axis=0) >= 10)

slope, intercept = np.polyfit(frame_idx[valid], mean_mag[valid], 1)
print(f'Drift growth rate: {slope:.4f} px/frame  ({slope*DPI*1000:.2f} mm/frame at dpi={DPI})')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(frame_idx[valid], mean_mag[valid], color='steelblue', linewidth=1.2, label='Mean |drift| across trajectories')
ax.plot(frame_idx[valid], slope * frame_idx[valid] + intercept, 'r--', linewidth=1, label=f'Linear fit: {slope:.4f} px/frame')
ax.set_xlabel('Frame index')
ax.set_ylabel('Mean |noise| (px)')
ax.set_title('Mean drift magnitude growth rate')
ax.legend()
plt.tight_layout()
plt.savefig('drift_growth_rate.png', dpi=150)
plt.show()

## 7. Trajectories overlaid on floorplan (GT vs VIO)

In [ ]:
floorplan = mpimg.imread(str(FLOORPLAN_PATH))

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(floorplan, origin='upper')

plot_names = rng.choice(list(noise_data.keys()), size=min(10, len(noise_data)), replace=False)
for name in plot_names:
    traj = noise_data[name]['traj']
    ax.plot(traj[:, GTX], traj[:, GTY], linewidth=1.2, alpha=0.8, label=f'{name} GT')
    ax.plot(traj[:, X],   traj[:, Y],   linewidth=0.8, alpha=0.5, linestyle='--')

ax.set_title('10 sample train trajectories: GT (solid) vs VIO (dashed)')
ax.set_xlabel('x (px)')
ax.set_ylabel('y (px)')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig('trajectories_on_floorplan.png', dpi=150)
plt.show()

## 8. Outlier detection

In [ ]:
# Flag trajectories where max drift > 3 * p95 across all trajectories
global_p95 = np.percentile([d['max_drift'] for d in noise_data.values()], 95)
threshold  = 3 * global_p95

outliers = {
    name: d for name, d in noise_data.items()
    if d['max_drift'] > threshold
}

print(f'Global p95 max drift: {global_p95:.1f} px')
print(f'Outlier threshold (3x p95): {threshold:.1f} px')
print(f'Outlier trajectories: {len(outliers)}')
for name, d in outliers.items():
    print(f'  {name}: max_drift={d["max_drift"]:.1f} px, mean_drift={d["mean_drift"]:.1f} px')

# Short trajectory check
short = {name: d for name, d in noise_data.items() if d['n_frames'] < 400}
print(f'\nTrajectories shorter than 400 frames: {len(short)}')
for name, d in short.items():
    print(f'  {name}: {d["n_frames"]} frames')

## 9. Smoke-test bounds for issue #12

After DPI scaling to Avalon (dpi=3.5), the expected noise magnitude is:

```
expected_mean_drift_avalon = mean_drift_universityA * (3.5 / 2.5)
```

The smoke test in #12 should assert `mean(|x - gt_x|)` is within a reasonable range of this value.

In [ ]:
AVALON_DPI = 3.5
scale = AVALON_DPI / DPI

global_mean   = all_mags.mean()
global_median = np.median(all_mags)
global_p95    = np.percentile(all_mags, 95)

print('=== Noise bounds for smoke test (#12) ===')
print(f'universityA (dpi={DPI}):')
print(f'  mean   |drift|: {global_mean:.1f} px')
print(f'  median |drift|: {global_median:.1f} px')
print(f'  p95    |drift|: {global_p95:.1f} px')
print()
print(f'Avalon (dpi={AVALON_DPI}, scale={scale:.2f}x):')
print(f'  expected mean   |drift|: {global_mean * scale:.1f} px')
print(f'  expected median |drift|: {global_median * scale:.1f} px')
print(f'  expected p95    |drift|: {global_p95 * scale:.1f} px')
print()
print('Smoke test assertion (issue #12):')
lo = global_mean * scale * 0.5
hi = global_p95  * scale * 1.5
print(f'  assert {lo:.0f} < mean(|x - gt_x|) < {hi:.0f}  (0.5x mean to 1.5x p95)')

## 10. Findings summary

Fill this in after running the notebook. Key questions to answer:
- What is the mean/median/p95 drift across train trajectories?
- How many frames does it take for drift to reach 50% of its final value (growth rate)?
- Is the noise autocorrelation length consistent with drift accumulation (random-walk-like)?
- How many outliers exist and should they be excluded from the noise library?
- What are the concrete smoke-test bounds for issue #12?